## 0. Setup

In [2]:
import subprocess
import sys

def _pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip_install(["dagshub", "mlflow", "pmdarima", "statsmodels"])

import os
import time
import zipfile
import warnings
import pickle
import tempfile

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pmdarima as pm
from joblib import Parallel, delayed
from sklearn.base import BaseEstimator, TransformerMixin

import mlflow
import mlflow.pyfunc
import dagshub

pd.set_option("display.max_columns", 60)

PIPELINE_START = time.time()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


## 1. DagsHub / MLflow init

In [3]:
dagshub.init(repo_owner="ikvas22", repo_name="Walmart-Recruiting---Store-Sales-Forecasting", mlflow=True)

MLFLOW_EXPERIMENT_NAME = "ARIMA_tarining"
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9b4a1dbc-40e4-4eae-b48f-3e9e91d22bce&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=a6a8d70b8f1b40da9f92cc0669a672b895397316e41c860b788aa2c80188db95




Accessing as ntsuk22

Initialized MLflow to track repo "ikvas22/Walmart-Recruiting---Store-Sales-Forecasting"

Repository ikvas22/Walmart-Recruiting---Store-Sales-Forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/1f0f5ee4856e4bfe84b922354f466938', creation_time=1783714908486, effective_trace_archival_retention=None, experiment_id='12', last_update_time=1783714908486, lifecycle_stage='active', name='ARIMA_tarining', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## 2. Data loading — Kaggle paths

In [4]:
KAGGLE_INPUT_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp/kaggle_working"
os.makedirs(WORK_DIR, exist_ok=True)


def _extract_zip(zip_name: str, extract_dir: str = WORK_DIR) -> str:
    """Extract a .zip that lives in the read-only Kaggle input dir into a
    writable working dir, returning the path to the extracted csv."""
    zip_path = os.path.join(KAGGLE_INPUT_DIR, zip_name)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    csv_name = zip_name.replace(".zip", "")
    return os.path.join(extract_dir, csv_name)


train_csv = _extract_zip("train.csv.zip")
features_csv = _extract_zip("features.csv.zip")
test_csv = _extract_zip("test.csv.zip")
sample_sub_csv = _extract_zip("sampleSubmission.csv.zip")
stores_csv = os.path.join(KAGGLE_INPUT_DIR, "stores.csv")  # not zipped

train_raw = pd.read_csv(train_csv, parse_dates=["Date"])
features_raw = pd.read_csv(features_csv, parse_dates=["Date"])
stores_raw = pd.read_csv(stores_csv)
test_raw = pd.read_csv(test_csv, parse_dates=["Date"])
sample_sub = pd.read_csv(sample_sub_csv)

df_merged = (
    train_raw
    .merge(features_raw, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores_raw, on=["Store"], how="left")
)
df_merged = df_merged.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

df_test_merged = (
    test_raw
    .merge(features_raw, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores_raw, on=["Store"], how="left")
)
df_test_merged = df_test_merged.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print(f"[data] train shape: {df_merged.shape}, date range "
      f"{df_merged['Date'].min().date()} -> {df_merged['Date'].max().date()}")
print(f"[data] store-dept combinations: {df_merged.groupby(['Store', 'Dept']).ngroups}")

[data] train shape: (421570, 16), date range 2010-02-05 -> 2012-10-26
[data] store-dept combinations: 3331


## 3. Exogenous feature builders (reused from feature_engineering notebook)

In [5]:
SUPERBOWL_DATES = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
LABORDAY_DATES = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
THANKSGIVING_DATES = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
CHRISTMAS_DATES = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X["Date"] = pd.to_datetime(X["Date"])
        week = X["Date"].dt.isocalendar().week.astype(int)
        X["week_sin"] = np.sin(2 * np.pi * week / 52)
        X["week_cos"] = np.cos(2 * np.pi * week / 52)
        X["is_thanksgiving"] = X["Date"].isin(THANKSGIVING_DATES).astype(int)
        X["is_christmas"] = X["Date"].isin(CHRISTMAS_DATES).astype(int)
        X["is_superbowl"] = X["Date"].isin(SUPERBOWL_DATES).astype(int)
        X["is_laborday"] = X["Date"].isin(LABORDAY_DATES).astype(int)
        X["IsHoliday"] = X["IsHoliday"].astype(int)
        return X


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        mdc = [c for c in ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
               if c in X.columns]
        if mdc:
            X["total_markdown_log"] = np.log1p(X[mdc].fillna(0).sum(axis=1))
        else:
            X["total_markdown_log"] = 0.0
        return X


def fourier_terms(dates: pd.Series, period: int = 52, order: int = 3) -> pd.DataFrame:
    t = (dates - dates.min()).dt.days.values / 7.0
    out = {}
    for k in range(1, order + 1):
        out[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        out[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(out, index=dates.index)


BASE_EXOG_COLS = [
    "week_sin", "week_cos", "is_thanksgiving", "is_christmas",
    "is_superbowl", "is_laborday", "total_markdown_log", "IsHoliday",
]
MINIMAL_EXOG_COLS = ["week_sin", "week_cos", "IsHoliday"]

VAL_START = "2012-04-01"   # kept identical to the tree-model / NeuralForecast notebooks
H = 4                      # forecast horizon, matches NeuralForecast setup
CV_FOLDS = 3               # rolling-origin folds for "strong" validation
N_SERIES = 15              # top-N (Store, Dept) pairs by mean sales; raise for a fuller run


def build_exog(df_merged: pd.DataFrame, fourier_order: int) -> tuple[pd.DataFrame, list]:
    calendar_tf = CalendarFeatureTransformer()
    markdown_tf = MarkdownFeatureTransformer()
    df_fe = calendar_tf.transform(df_merged)
    df_fe = markdown_tf.transform(df_fe)
    fcols = fourier_terms(df_fe["Date"], period=52, order=fourier_order)
    df_fe = pd.concat([df_fe, fcols], axis=1)
    exog_full = BASE_EXOG_COLS + list(fcols.columns)
    return df_fe, exog_full


def get_series(df: pd.DataFrame, store: int, dept: int, exog_cols: list) -> pd.DataFrame:
    s = (
        df[(df["Store"] == store) & (df["Dept"] == dept)]
        .sort_values("Date")
        .set_index("Date")
    )
    s = s.asfreq("W-FRI")
    s["Weekly_Sales"] = s["Weekly_Sales"].interpolate(limit_direction="both")
    for c in exog_cols:
        s[c] = s[c].ffill().bfill()
    return s

## 4. Rolling-origin CV fit/forecast for one (Store, Dept) series

In [6]:
def fit_and_forecast_one_series_cv(df_fe: pd.DataFrame, store: int, dept: int,
                                    exog_cols: list, val_start: str, h: int,
                                    cv_folds: int, arima_kwargs: dict) -> dict | None:
    """
    Rolling-origin CV: fold 0 uses [val_start, val_start+h) as the val window,
    fold 1 shifts the origin back by h weeks, etc. Each fold re-fits
    auto_arima on all data strictly before its own origin (no leakage).
    """
    s = get_series(df_fe, store, dept, exog_cols)
    val_start_ts = pd.Timestamp(val_start)

    fold_records = []
    last_model = None
    for fold in range(cv_folds):
        origin = val_start_ts - pd.Timedelta(weeks=h * fold)
        s_train = s[s.index < origin]
        s_val = s[s.index >= origin].iloc[:h]

        if len(s_train) < 60 or len(s_val) < 1:
            continue

        y_train = s_train["Weekly_Sales"]
        exog_train = s_train[exog_cols]
        exog_val = s_val[exog_cols]

        try:
            model = pm.auto_arima(y_train, exogenous=exog_train, **arima_kwargs)
            preds = model.predict(n_periods=len(s_val), exogenous=exog_val)
        except Exception:
            continue

        weights = np.where(s_val["IsHoliday"].values, 5.0, 1.0)
        wmae = np.sum(weights * np.abs(s_val["Weekly_Sales"].values - preds)) / np.sum(weights)
        fold_records.append({"fold": fold, "order": model.order, "wmae": wmae, "n_train": len(s_train)})
        last_model = model  # keep the most recent (most data) fit for the registry artifact

    if not fold_records:
        return None

    return {
        "store": store, "dept": dept,
        "fold_records": fold_records,
        "mean_wmae": float(np.mean([f["wmae"] for f in fold_records])),
        "n_folds": len(fold_records),
        "final_model": last_model,
    }

## 5. Custom MLflow pyfunc wrapper — bundles all per-series ARIMA models

In [7]:
class ARIMAStoreDeptWrapper(mlflow.pyfunc.PythonModel):
    """
    Wraps a dict of {(store, dept): fitted pmdarima model} plus the exog
    column list used to fit them. `predict` expects a DataFrame with
    columns: Store, Dept, plus all exog columns, one row per forecast step
    (rows for the same Store/Dept must be contiguous and in date order).
    """

    def load_context(self, context):
        with open(context.artifacts["models_pkl"], "rb") as f:
            bundle = pickle.load(f)
        self.models = bundle["models"]
        self.exog_cols = bundle["exog_cols"]

    def predict(self, context, model_input: pd.DataFrame, params=None):
        out = []
        for (store, dept), grp in model_input.groupby(["Store", "Dept"]):
            key = (store, dept)
            if key not in self.models:
                out.append(pd.Series(np.nan, index=grp.index))
                continue
            model = self.models[key]
            exog = grp[self.exog_cols]
            preds = model.predict(n_periods=len(grp), exogenous=exog)
            out.append(pd.Series(preds, index=grp.index))
        return pd.concat(out).sort_index()

## 6. Hyperparameter grid — 3 experiments

In [8]:
HYPERPARAM_CONFIGS = [
    {
        "name": "exp1_baseline",
        "fourier_order": 3,
        "exog_set": "full",
        "arima_kwargs": dict(seasonal=False, stepwise=True, suppress_warnings=True,
                              error_action="ignore", max_p=5, max_q=5, max_d=2,
                              information_criterion="aic"),
    },
    {
        "name": "exp3_bic_criterion",
        "fourier_order": 3,
        "exog_set": "full",
        "arima_kwargs": dict(seasonal=False, stepwise=True, suppress_warnings=True,
                              error_action="ignore", max_p=5, max_q=5, max_d=2,
                              information_criterion="bic"),
    },
    {
        "name": "exp5_minimal_exog",
        "fourier_order": 3,
        "exog_set": "minimal",
        "arima_kwargs": dict(seasonal=False, stepwise=True, suppress_warnings=True,
                              error_action="ignore", max_p=5, max_q=5, max_d=2,
                              information_criterion="aic"),
    },
]


## 7. Main experiment loop

In [9]:
vol = (
    df_merged.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
    .sort_values(ascending=False)
)
PAIRS = list(vol.index[:N_SERIES])
print(f"[cv] using {len(PAIRS)} (Store, Dept) pairs, {CV_FOLDS} rolling folds, horizon={H}")

experiment_summaries = []  # one dict per experiment, used to pick the winner

for cfg in HYPERPARAM_CONFIGS:
    exp_name = cfg["name"]
    print(f"\n===== {exp_name} =====")

    with mlflow.start_run(run_name=exp_name) as parent_run:
        mlflow.log_params({
            "fourier_order": cfg["fourier_order"],
            "exog_set": cfg["exog_set"],
            **{f"arima__{k}": v for k, v in cfg["arima_kwargs"].items()},
        })

        # ---- 7a. preprocessing run --------------------------------------
        with mlflow.start_run(run_name="preprocessing", nested=True):
            t0 = time.time()
            mlflow.log_param("train_shape", str(df_merged.shape))
            mlflow.log_param("date_min", str(df_merged["Date"].min().date()))
            mlflow.log_param("date_max", str(df_merged["Date"].max().date()))
            mlflow.log_param("n_store_dept_pairs_total", df_merged.groupby(["Store", "Dept"]).ngroups)
            mlflow.log_param("val_start", VAL_START)
            mlflow.log_param("horizon_weeks", H)
            mlflow.log_metric("preprocessing_seconds", time.time() - t0)

        # ---- 7b. feature-engineering run ---------------------------------
        with mlflow.start_run(run_name="feature-engineering", nested=True):
            t0 = time.time()
            df_fe, exog_full_cols = build_exog(df_merged, cfg["fourier_order"])
            mlflow.log_param("fourier_order", cfg["fourier_order"])
            mlflow.log_param("n_exog_full", len(exog_full_cols))
            mlflow.log_param("exog_full_cols", str(exog_full_cols))
            mlflow.log_metric("feature_engineering_seconds", time.time() - t0)

        # ---- 7c. feature-selection run ------------------------------------
        with mlflow.start_run(run_name="feature-selection", nested=True):
            t0 = time.time()
            if cfg["exog_set"] == "minimal":
                exog_cols = [c for c in MINIMAL_EXOG_COLS if c in df_fe.columns]
            else:
                exog_cols = exog_full_cols
            mlflow.log_param("exog_set_choice", cfg["exog_set"])
            mlflow.log_param("selected_exog_cols", str(exog_cols))
            mlflow.log_param("n_selected", len(exog_cols))
            mlflow.log_param("rationale",
                              "SARIMAX needs no IV/MI/corr filter like tree models; "
                              "selection here means choosing between the full exogenous "
                              "set and a minimal core set to test overfitting risk.")
            mlflow.log_metric("feature_selection_seconds", time.time() - t0)

        # ---- 7d. training run (CV + fit) ----------------------------------
        with mlflow.start_run(run_name="training", nested=True) as training_run:
            t0 = time.time()
            mlflow.log_params({f"arima__{k}": v for k, v in cfg["arima_kwargs"].items()})
            mlflow.log_param("cv_folds", CV_FOLDS)
            mlflow.log_param("n_series", len(PAIRS))

            results = Parallel(n_jobs=-1, verbose=0)(
                delayed(fit_and_forecast_one_series_cv)(
                    df_fe, store, dept, exog_cols, VAL_START, H, CV_FOLDS, cfg["arima_kwargs"]
                )
                for store, dept in PAIRS
            )
            results = [r for r in results if r is not None]

            all_fold_wmaes = [f["wmae"] for r in results for f in r["fold_records"]]
            mean_cv_wmae = float(np.mean(all_fold_wmaes)) if all_fold_wmaes else float("inf")
            std_cv_wmae = float(np.std(all_fold_wmaes)) if all_fold_wmaes else float("nan")
            n_fitted = len(results)

            mlflow.log_metric("mean_cv_wmae", mean_cv_wmae)
            mlflow.log_metric("std_cv_wmae", std_cv_wmae)
            mlflow.log_metric("n_series_fitted", n_fitted)
            mlflow.log_metric("training_seconds", time.time() - t0)

            # per-series summary as an artifact (not one of the "5 required"
            # runs, just supporting evidence attached to the training run)
            per_series_df = pd.DataFrame([
                {"Store": r["store"], "Dept": r["dept"], "mean_wmae": r["mean_wmae"], "n_folds": r["n_folds"]}
                for r in results
            ]).sort_values("mean_wmae")
            tmp_csv = os.path.join(tempfile.gettempdir(), f"{exp_name}_per_series_wmae.csv")
            per_series_df.to_csv(tmp_csv, index=False)
            mlflow.log_artifact(tmp_csv)

            # bundle models + exog cols, log as a (not-yet-registered) pyfunc model
            models_bundle = {
                "models": {(r["store"], r["dept"]): r["final_model"] for r in results if r["final_model"] is not None},
                "exog_cols": exog_cols,
            }
            tmp_pkl = os.path.join(tempfile.gettempdir(), f"{exp_name}_models.pkl")
            with open(tmp_pkl, "wb") as f:
                pickle.dump(models_bundle, f)

            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=ARIMAStoreDeptWrapper(),
                artifacts={"models_pkl": tmp_pkl},
            )

            print(f"[{exp_name}] mean_cv_wmae={mean_cv_wmae:,.2f}  (n_series_fitted={n_fitted})")

            experiment_summaries.append({
                "name": exp_name,
                "parent_run_id": parent_run.info.run_id,
                "training_run_id": training_run.info.run_id,
                "mean_cv_wmae": mean_cv_wmae,
                "exog_cols": exog_cols,
            })

[cv] using 15 (Store, Dept) pairs, 3 rolling folds, horizon=4

===== exp1_baseline =====
🏃 View run preprocessing at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/ea31b8fada534a89941ec6ee695b1c0a
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run feature-engineering at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/cd6d32d3e07d4579b82a92ecab5b05e3
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run feature-selection at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/cba4fd1f313f476e93d8d6e5d9d2ac30
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12


2026/07/11 03:30:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 03:30:28 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/11 03:30:28 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


[exp1_baseline] mean_cv_wmae=11,704.42  (n_series_fitted=15)
🏃 View run training at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/d0b25deae86f4128be084031f3f71622
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run exp1_baseline at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/d991ccd12bdb4ce1b9157baff23a369f
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12

===== exp3_bic_criterion =====
🏃 View run preprocessing at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/81987f6cb4504c80a842e1228d30c813
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run feature-engineering at: https://dagshub.com/ikvas22/

2026/07/11 03:31:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 03:31:55 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/11 03:31:55 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


[exp3_bic_criterion] mean_cv_wmae=11,522.74  (n_series_fitted=15)
🏃 View run training at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/a2a904e5a0724b1fbc87b61a283abc3a
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run exp3_bic_criterion at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/63ea9edc3d2a411db8ea5a9fb7fc8c8f
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12

===== exp5_minimal_exog =====
🏃 View run preprocessing at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/05d3c456c4fc42d4bfb64d9422d6566a
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run feature-engineering at: https://dagshub.com

2026/07/11 03:33:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 03:33:41 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/07/11 03:33:41 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.


[exp5_minimal_exog] mean_cv_wmae=11,704.42  (n_series_fitted=15)
🏃 View run training at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/c550523e139f48a2a73783d61de438d1
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
🏃 View run exp5_minimal_exog at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/0b96bbfc7cde41978250f333a4b44f72
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12


## 8. Pick the best experiment and register it

In [11]:
best = min(experiment_summaries, key=lambda d: d["mean_cv_wmae"])
print(f"\n[best] {best['name']}  mean_cv_wmae={best['mean_cv_wmae']:,.2f}")

REGISTERED_MODEL_NAME = "ARIMA_Walmart_SalesForecast"
client = mlflow.tracking.MlflowClient()

# Ensure the registered model exists (ignore if it already does, e.g. a
# previous run of this notebook already created it)
try:
    client.create_registered_model(REGISTERED_MODEL_NAME)
except mlflow.exceptions.MlflowException:
    pass

# NOTE: mlflow.register_model(model_uri="runs:/<id>/model") internally looks
# up a "logged model" entity via _get_logged_models_from_run(), which some
# tracking servers (DagsHub included) don't populate the same way -> raises
# "Unable to find a logged_model with artifact_path model under run ...".
# Registering directly against the run's artifact_uri sidesteps that lookup
# and works reliably against DagsHub's MLflow-compatible backend.
best_run_info = mlflow.get_run(best["training_run_id"]).info
model_source = f"{best_run_info.artifact_uri}/model"

registered_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=model_source,
    run_id=best["training_run_id"],
)
print(f"[registry] registered '{REGISTERED_MODEL_NAME}' version {registered_version.version} "
      f"from run {best['training_run_id']} (source={model_source})")

# also tag the winning parent run so it's easy to find in the DagsHub UI
with mlflow.start_run(run_id=best["parent_run_id"]):
    mlflow.set_tag("is_best_experiment", "true")
    mlflow.log_metric("final_selected_mean_cv_wmae", best["mean_cv_wmae"])


[best] exp3_bic_criterion  mean_cv_wmae=11,522.74


2026/07/11 03:40:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ARIMA_Walmart_SalesForecast, version 1


[registry] registered 'ARIMA_Walmart_SalesForecast' version 1 from run a2a904e5a0724b1fbc87b61a283abc3a (source=mlflow-artifacts:/1f0f5ee4856e4bfe84b922354f466938/a2a904e5a0724b1fbc87b61a283abc3a/artifacts/model)
🏃 View run exp3_bic_criterion at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12/runs/63ea9edc3d2a411db8ea5a9fb7fc8c8f
🧪 View experiment at: https://dagshub.com/ikvas22/Walmart-Recruiting---Store-Sales-Forecasting.mlflow/#/experiments/12
